# Bounded machine-learning survival models

This notebook compares a Random Survival Forest (RSF) and Gradient Boosted Survival model on a bounded 2018 cohort. Both models use right-censored outcomes through scikit-survival.

The run is a reproducible smoke analysis, not a full-cohort estimate. It uses 1,000 rows per state, a fixed stratified train/test split, and a small number of estimators.

In [ ]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ml_survival import (
    concordance_index,
    feature_importance_table,
    fit_gradient_boosted_survival,
    fit_random_survival_forest,
)

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cases_clean.parquet'
ROWS_PER_STATE = 1000
RANDOM_SEED = 19
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(PROCESSED_PATH)
print({'processed_path': str(PROCESSED_PATH), 'rows_per_state': ROWS_PER_STATE})


## Bounded cohort

DuckDB selects 1,000 rows per state from 2018. The full parquet is not loaded into pandas.

In [ ]:
query = """
WITH ranked AS (
    SELECT
        duration, event, state_code, filing_month, criminal,
        bailable_ipc, number_sections_ipc,
        ROW_NUMBER() OVER (PARTITION BY state_code ORDER BY ddl_case_id) AS row_number
    FROM read_parquet(?)
    WHERE year = 2018
)
SELECT duration, event, state_code, filing_month, criminal,
       bailable_ipc, number_sections_ipc
FROM ranked
WHERE row_number <= ?
ORDER BY state_code, row_number
"""
with duckdb.connect() as con:
    sample = con.execute(query, [str(PROCESSED_PATH), ROWS_PER_STATE]).fetchdf()

assert len(sample) == 5000
assert sample['state_code'].nunique() == 5
print({
    'rows': len(sample),
    'events': int(sample['event'].sum()),
    'censored': int((sample['event'] == 0).sum()),
    'states': sorted(sample['state_code'].unique()),
})


## Features and fixed split

State is one-hot encoded. Numeric missing feature values are filled with zero for this smoke run. The split is stratified by the event indicator and is used only for bounded discrimination checks.

In [ ]:
state_features = pd.get_dummies(sample['state_code'], prefix='state', dtype=float)
numeric = sample[['filing_month', 'criminal', 'bailable_ipc', 'number_sections_ipc']].apply(
    pd.to_numeric, errors='coerce'
).fillna(0.0)
X = pd.concat([state_features, numeric], axis=1).astype(float)
duration = pd.to_numeric(sample['duration'], errors='coerce')
event = pd.to_numeric(sample['event'], errors='coerce').astype(int)
train_idx, test_idx = train_test_split(
    np.arange(len(sample)), test_size=0.25, random_state=RANDOM_SEED, stratify=event
)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
duration_train, duration_test = duration.iloc[train_idx], duration.iloc[test_idx]
event_train, event_test = event.iloc[train_idx], event.iloc[test_idx]
print({'features': list(X.columns), 'train_rows': len(X_train), 'test_rows': len(X_test)})


## Fit the two survival models

The estimator counts are intentionally small to keep this notebook a smoke test.

In [ ]:
forest = fit_random_survival_forest(
    X_train, duration_train, event_train,
    n_estimators=40, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=1,
)
boosted = fit_gradient_boosted_survival(
    X_train, duration_train, event_train,
    n_estimators=40, learning_rate=0.05, max_depth=2, random_state=RANDOM_SEED,
)
print({'forest': type(forest).__name__, 'boosted': type(boosted).__name__})


## Test-set discrimination and feature importance

Harrell's C-index accounts for censoring. scikit-survival does not implement Random Survival Forest feature importances, so the importance table uses the boosted model.

In [ ]:
forest_c_index = concordance_index(forest, X_test, duration_test, event_test)
boosted_c_index = concordance_index(boosted, X_test, duration_test, event_test)
importance = feature_importance_table(boosted, list(X.columns))
assert np.isfinite([forest_c_index, boosted_c_index]).all()
assert 0 <= forest_c_index <= 1 and 0 <= boosted_c_index <= 1
print({
    'forest_test_c_index': round(forest_c_index, 6),
    'boosted_test_c_index': round(boosted_c_index, 6),
})
print(importance.to_string(index=False))


## Interpretation boundary

These are bounded smoke-test results, not full-cohort estimates, causal effects, or final feature rankings. The models use a small fixed split and a limited feature set. Any substantive ML comparison would need a prespecified evaluation design, broader feature review, and repeated or cross-validated assessment.